# Week 99, Day 2: System design

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w099d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
base, cap = 1, 30
print([min(cap, base * 2 ** i) for i in range(7)])

> **Key idea.** Retry jobs with capped exponential backoff plus jitter. Expect duplicates (at-least-once delivery): make handlers idempotent with keys. Visibility timeouts must exceed the slowest normal job; unacknowledged jobs reappear. After a few failed attempts, dead-letter the job. Size workers with Little's law at about 70%, counting retries. Age priorities to prevent starvation.

### Exercise: Backoff with jitter

Write `backoff(attempts, base, cap, jitter)`: the delays before each retry `i = 0, 1, ...`, each `min(cap, base * 2 ** i)` passed through `jitter` (a function of one delay), rounded to 2 decimal places.

In [ ]:
def backoff(attempts, base, cap, jitter):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\n\ndef test_backoff():\n    \"capped, doubled, jittered\"\n    assert backoff(6, 1, 20, lambda d: d) == [1, 2, 4, 8, 16, 20]\n    assert backoff(3, 0.5, 10, lambda d: d / 2) == [0.25, 0.5, 1.0]\n    rng = random.Random(1)\n    delays = backoff(5, 1, 30, lambda d: rng.uniform(0, d))\n    assert all(0 <= d <= c for d, c in zip(delays, [1, 2, 4, 8, 16])) and len(set(delays)) == 5")

### Exercise: Harmless duplicates

Write a class `Idempotent(handler, clock, keep_seconds)`: `run(key, payload)` calls `handler(payload)` the first time a key is seen and returns its result; a repeat of the key within `keep_seconds` (by `clock()`) returns the stored result without calling the handler. Keys older than `keep_seconds` are forgotten (and a later repeat runs again). `calls` counts handler calls.

In [ ]:
class Idempotent:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_idempotent():\n    \"once, however many deliveries\"\n    now = [0]\n    sent = []\n    def send_email(payload):\n        sent.append(payload)\n        return f\"sent to {payload}\"\n    job = Idempotent(send_email, lambda: now[0], keep_seconds=3600)\n    assert job.run(\"k1\", \"ada@example.org\") == \"sent to ada@example.org\"\n    assert job.run(\"k1\", \"ada@example.org\") == \"sent to ada@example.org\" and job.calls == 1\n    job.run(\"k2\", \"bo@example.org\")\n    now[0] = 4000\n    job.run(\"k1\", \"ada@example.org\")\n    assert job.calls == 3 and sent == [\"ada@example.org\", \"bo@example.org\", \"ada@example.org\"]")

### Exercise: Timeouts and the dead-letter queue

Write `deliveries(jobs, timeout, max_receives)`: each job is `(name, seconds, crashes)`: it takes `seconds` to process, and its first `crashes` attempts crash before acknowledging. An attempt succeeds if it doesn't crash and `seconds <= timeout`; otherwise the job reappears. Return `"NAME: delivered N, done"` or, once it has been received `max_receives` times without success, `"NAME: delivered N, dead-lettered"`.

In [ ]:
def deliveries(jobs, timeout, max_receives):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deliveries():\n    \"at least once\"\n    jobs = [(\"resize-1\", 4, 0), (\"resize-2\", 4, 2), (\"report\", 90, 0), (\"email\", 1, 5)]\n    assert deliveries(jobs, timeout=30, max_receives=4) == [\n        \"resize-1: delivered 1, done\", \"resize-2: delivered 3, done\", \"report: delivered 4, dead-lettered\", \"email: delivered 4, dead-lettered\"]\n    assert deliveries([(\"x\", 30, 0)], 30, 1) == [\"x: delivered 1, done\"]")

### Exercise: How many workers?

Write `workers_needed(jobs_per_s, seconds_each, failure_rate=0.0, max_receives=1, utilisation=0.7)`: attempts per job are `1 + p + p² + ...` up to `max_receives` terms (p the failure rate); busy workers are jobs a second × attempts × seconds each; return the whole number of workers needed at the given utilisation (rounded up), at least 1.

In [ ]:
import math

def workers_needed(jobs_per_s, seconds_each, failure_rate=0.0, max_receives=1, utilisation=0.7):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_workers():\n    \"Little's law, with retries\"\n    assert workers_needed(10, 2) == 29 and workers_needed(10, 2, utilisation=1) == 20\n    assert workers_needed(10, 2, failure_rate=0.5, max_receives=3, utilisation=1) == 35\n    assert workers_needed(0.01, 1) == 1")

### Exercise: No job waits for ever

Write `run_order(jobs, now, ageing)`: each job is `(name, priority, enqueued)` where a smaller priority number is more urgent. Its effective priority is `priority - (now - enqueued) // ageing`. Return the names in the order they'd run: effective priority, then enqueue time, then name.

In [ ]:
def run_order(jobs, now, ageing):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"old jobs rise\"\n    jobs = [(\"stats\", 5, 0), (\"email\", 1, 95), (\"resize\", 2, 90), (\"backup\", 5, 60)]\n    assert run_order(jobs, now=100, ageing=20) == [\"stats\", \"email\", \"resize\", \"backup\"]\n    assert run_order(jobs, now=100, ageing=1000) == [\"email\", \"resize\", \"stats\", \"backup\"]")

### Bug hunt: A wait that never stopped growing

Without a cap, the seventh retry waits over a minute and the tenth over eight minutes: this prints `512`. Cap each delay at 30 seconds so it prints `30`.

In [ ]:
base = 1
delays = [base * 2 ** i for i in range(10)]
print(delays[-1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"min(cap, ...)\"\n    expect_output(\"30\")")

### Bug hunt: An email sent twice

The handler never records which keys it has finished, so a redelivered job sends the email again and this prints `2`. Record finished keys and skip them so it prints `1`.

In [ ]:
sent = []
done = set()

def handle(key, address):
    sent.append(address)

for key, address in [("k1", "ada@example.org"), ("k1", "ada@example.org")]:
    handle(key, address)
print(len(sent))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"idempotency keys\"\n    expect_output(\"1\")")

### Puzzle: Receive until done or dead

Arrange the loop for one job in `deliveries`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the delivery guarantee most queues give (three words, hyphens between)
q = ___   # attempts per job with a failure rate of 0.5 and up to 2 receives (a float)
r = ___   # True or False: the visibility timeout should be shorter than the slowest normal job
s = ___   # where a job goes after too many failed attempts (three words)
t = ___   # the word for raising a waiting job's priority over time

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"at-least-once\" and q == 1.5 and r is False, \"p, q and r\"\n    assert s == \"dead-letter queue\" and t == \"ageing\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a queue plan

Write `queue_plan(req)`. `req` has `jobs_per_s`, `seconds_each`, `failure_rate` (the chance an attempt fails), `max_receives`, `workers`, `timeout` and `slowest_seconds`. Return `(lines, warnings)`:

- `"attempts per job: A"` (`1 + p + ... + p^(m-1)`, 3 decimal places);
- `"dead-lettered: D%"` (`p^m` as a percentage, 3 decimal places);
- `"busy workers: B of W"` (jobs × attempts × seconds, 1 decimal place);
- `"utilisation: U%"` (busy over workers, 1 decimal place);
- `"backlog growth: G jobs a second"` (attempts arriving a second minus what the workers finish, `jobs × attempts − workers / seconds`, never below 0, 1 decimal place).

`warnings` lists, in this order, `"visibility timeout shorter than the slowest job"` if `timeout < slowest_seconds`, `"over 70% utilisation"` if it is, and `"queue grows without limit"` if the backlog growth is above 0.

In [ ]:
def queue_plan(req):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("REQ = {\"jobs_per_s\": 40, \"seconds_each\": 0.5, \"failure_rate\": 0.1, \"max_receives\": 4, \"workers\": 30, \"timeout\": 30, \"slowest_seconds\": 12}\n\ndef test_plan():\n    \"comfortable\"\n    lines, warnings = queue_plan(REQ)\n    assert lines == [\"attempts per job: 1.111\", \"dead-lettered: 0.010%\", \"busy workers: 22.2 of 30\", \"utilisation: 74.1%\",\n                     \"backlog growth: 0.0 jobs a second\"]\n    assert warnings == [\"over 70% utilisation\"]\n\ndef test_trouble():\n    \"everything wrong\"\n    lines, warnings = queue_plan({**REQ, \"workers\": 20, \"timeout\": 10})\n    assert lines[-1] == \"backlog growth: 4.4 jobs a second\"\n    assert warnings == [\"visibility timeout shorter than the slowest job\", \"over 70% utilisation\", \"queue grows without limit\"]\n    assert queue_plan({**REQ, \"workers\": 100})[1] == []")

## Warm-up

Doubling the wait after each failure (1, 2, 4, 8 seconds...) gives a struggling service room to recover; capping it stops the waits growing absurdly. Week 91 retried calls; a job queue retries whole jobs, and needs a little randomness too, so a thousand failed jobs don't all come back in the same second.

## What goes wrong

- **A worker crashes mid-job.** The queue must notice and give the job to another worker.
- **The same job runs twice.** Retries and crashes make duplicates unavoidable, so jobs must be **idempotent**: running twice has the same effect as once. Give each job an **idempotency key** and record finished keys.
- **A job always fails** (bad input). Retrying forever wastes workers; after a few attempts move it to a **dead-letter queue** for a person to inspect.
- **Arrivals outpace workers.** The backlog grows by the difference every second (Week 98's queues).
- **Low-priority jobs never run** while high-priority ones keep arriving: **starvation**. Raise a job's priority as it waits (**ageing**).

## Delivery guarantees

Most queues (Amazon SQS, RabbitMQ, Redis-based ones) give **at-least-once** delivery: a worker **receives** a job, which becomes invisible for a **visibility timeout**; if the worker **acknowledges** it in time, it's deleted; if not (a crash, or a job slower than the timeout), it reappears for another worker. That's why duplicates happen, and why the timeout must be longer than the slowest normal job. "Exactly once" is achieved, in practice, as at-least-once delivery plus idempotent handling.

## Sizing and fairness

Workers needed: Little's law again: busy workers = jobs a second × seconds a job; at 70% utilisation, divide by 0.7 and round up. Retries add load: if a fraction p of attempts fail, each job takes on average 1 + p + p² + ... attempts (up to the limit). Priority: a heap ordered by priority, then arrival; ageing subtracts from the priority number as time passes so old jobs eventually rise.

## Practice

## Quiz

## Daily challenge

## A mock round

Forty-five minutes on **"Design the system that recalculates every Waymark user's statistics each night"**: requirements, estimates with `queue_plan`, the queue and workers, idempotency keys, what happens when the statistics job for one user always fails, and how you'd watch it.

## Before you go

Tomorrow is a Python deep-dive interview: questions about the language itself, from the data model to generators, the GIL and memory.